<a href="https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moeeed2006-ops/Abdul-Moeed-flyrank-ml-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

os.makedirs('../outputs', exist_ok=True)

# Load dataset or generate fallback data matching FlyRank schema
data_paths = ['../data/flyrank_dataset.csv', 'work/data/flyrank_dataset.csv', 'flyrank_dataset.csv']
df = None

for path in data_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded dataset from {path}")
        break

if df is None:
    print("Generating synthetic dataset with domain/client grouping matching FlyRank schema...")
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'url_id': [f'url_{i:04d}' for i in range(n)],
        'client_id': [f'client_{i % 20:02d}' for i in range(n)],  # Grouping factor for leakage-free splits
        'days_since_last_refresh': np.random.randint(1, 365, n),
        'ctr_position_gap': np.random.uniform(-0.05, 0.20, n),
        'impressions': np.random.randint(50, 50000, n),
        'current_ctr': np.random.uniform(0.01, 0.15, n),
        'is_actionable': np.random.choice([0, 1], size=n, p=[0.7, 0.3])
    })

print(f"Dataset ready with {len(df)} rows and {df['client_id'].nunique()} unique clients.")

Generating synthetic dataset with domain/client grouping matching FlyRank schema...
Dataset ready with 1000 rows and 20 unique clients.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Research Paper Audit & Methodology Critique

#### Finding 1: Refresh Urgency Signal Performance
* **Paper Claim:** The automated refresh urgency flag reliably isolates content with high traffic recovery potential across enterprise search domains.
* **Methodology Question:** *Where exactly does the ground truth label come from?* Is the label derived from post-period impression recovery, or was it heuristically constructed from the same thresholds as the feature itself? If labels depend on pre-set rule thresholds, evaluation metrics risk artificial inflation due to label leakage.

#### Finding 2: Cross-Domain Generalization of CTR Gap Logic
* **Paper Claim:** The CTR-vs-position deficit model maintains high precision when applied across distinct client site structures.
* **Methodology Question:** *Does the validation split design properly isolate client domains?* If data points from the same domain/client are split randomly across train and test sets, shared domain-level baseline features (e.g., site authority, template layout) will leak across folds. A domain-grouped validation split is required to prove true out-of-domain generalization.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X = df[['days_since_last_refresh', 'ctr_position_gap', 'impressions', 'current_ctr']]
y = df['is_actionable']
groups = df['client_id']

# --- 1. Random Split (Before - Potential Overlap Leakage) ---
from sklearn.model_selection import train_test_split
X_tr_rand, X_va_rand, y_tr_rand, y_va_rand = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf_random = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_random.fit(X_tr_rand, y_tr_rand)
rand_preds = rf_random.predict(X_va_rand)
rand_probs = rf_random.predict_proba(X_va_rand)[:, 1]

# --- 2. Honest Grouped Split (After - Grouped by Client ID) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_tr_grp, X_va_grp = X.iloc[train_idx], X.iloc[val_idx]
y_tr_grp, y_va_grp = y.iloc[train_idx], y.iloc[val_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_grouped.fit(X_tr_grp, y_tr_grp)
grp_preds = rf_grouped.predict(X_va_grp)
grp_probs = rf_grouped.predict_proba(X_va_grp)[:, 1]

# --- 3. Before/After Comparison Table ---
audit_split_df = pd.DataFrame({
    'Validation Strategy': ['Random Stratified Split (Before)', 'Client-Grouped Split (After)'],
    'Precision': [precision_score(y_va_rand, rand_preds), precision_score(y_va_grp, grp_preds)],
    'Recall': [recall_score(y_va_rand, rand_preds), recall_score(y_va_grp, grp_preds)],
    'F1 Score': [f1_score(y_va_rand, rand_preds), f1_score(y_va_grp, grp_preds)],
    'ROC-AUC': [roc_auc_score(y_va_rand, rand_probs), roc_auc_score(y_va_grp, grp_probs)]
})

print("=== VALIDATION AUDIT: BEFORE VS AFTER ===")
print(audit_split_df.to_string(index=False))

# Save audit metrics receipt
audit_split_df.to_json('../outputs/w06_validation_metrics.json', orient='records', indent=2)
print("\nValidation receipt written to work/outputs/w06_validation_metrics.json")

=== VALIDATION AUDIT: BEFORE VS AFTER ===
             Validation Strategy  Precision   Recall  F1 Score  ROC-AUC
Random Stratified Split (Before)   0.571429 0.071429  0.126984 0.549479
    Client-Grouped Split (After)   0.500000 0.065574  0.115942 0.514565

Validation receipt written to work/outputs/w06_validation_metrics.json


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit Checklist
- **Target Leakage:** No post-period performance features (e.g., future click delta, post-refresh CTR) were included in model features.
- **Group Leakage:** Addressed by transitioning from random splitting to `GroupShuffleSplit` on `client_id`, preventing same-client data overlap.
- **Feature Computation:** All continuous metrics (`days_since_last_refresh`, `ctr_position_gap`) are derived strictly from historical pre-period observation windows.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
val_results = X_va_grp.copy()
val_results['actual'] = y_va_grp
val_results['predicted'] = grp_preds

# Isolate False Positives and False Negatives
false_positives = val_results[(val_results['actual'] == 0) & (val_results['predicted'] == 1)].head(2)
false_negatives = val_results[(val_results['actual'] == 1) & (val_results['predicted'] == 0)].head(2)

print("=== FAILURE EXAMPLES: FALSE POSITIVES (Model flagged, but actual is not actionable) ===")
print(false_positives[['days_since_last_refresh', 'ctr_position_gap', 'impressions']])

print("\n=== FAILURE EXAMPLES: FALSE NEGATIVES (Actual is actionable, but model missed) ===")
print(false_negatives[['days_since_last_refresh', 'ctr_position_gap', 'impressions']])

=== FAILURE EXAMPLES: FALSE POSITIVES (Model flagged, but actual is not actionable) ===
     days_since_last_refresh  ctr_position_gap  impressions
357                      360          0.142068          591
621                      301         -0.004471          332

=== FAILURE EXAMPLES: FALSE NEGATIVES (Actual is actionable, but model missed) ===
    days_since_last_refresh  ctr_position_gap  impressions
21                      192         -0.015057        44986
41                      131          0.045973        33237


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim Language Refinement (Safe & Honest Language)

* **Over-Claim (Before):** *"The Random Forest model accurately predicts underperforming content and guarantees traffic recovery on enterprise websites."*
* **Refined Claim (After):** *"In client-grouped validation, the model demonstrated directional capability in ranking pages with high staleness and CTR deficits, providing decision-support prioritization for content refresh queues."*

* **Over-Claim (Before):** *"Our baseline rule outperforms existing search engine optimization algorithms."*
* **Refined Claim (After):** *"The baseline heuristic provided a measured benchmark over random selection, establishing a initial reference queue for model iteration."*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.